# GAN using Tensorflow

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os

os.makedirs('samples/', exist_ok=True)
# enable GPU
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [2]:
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
# 0~1 normalization
train_images, test_images = train_images / 255.0, test_images / 255.0
NUM_DIGITS = 10
train_labels = tf.keras.utils.to_categorical(train_labels, NUM_DIGITS)
test_labels = tf.keras.utils.to_categorical(test_labels, NUM_DIGITS)

# model setting
total_epoch = 200
batch_size = 100
learning_rate = 0.0002

# hidden layer setting
n_hidden = 256
n_input = 28*28
n_noise = 128

# reshape for FC
train_images_batch = train_images.reshape(-1, batch_size, 28*28)
train_labels_batch = train_labels.reshape(-1, batch_size, 10)
test_images_batch = test_images.reshape(-1, 28*28)
train_samples = train_images.shape[0]
steps = train_samples // batch_size

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [3]:
# Neural Network Model
# Unsupervised Learning, without using y
X = tf.keras.Input(shape=(n_input,))
# Noise Z
Z = tf.keras.Input(shape=(n_noise,))
# Generator Network
G = tf.keras.Sequential([
    tf.keras.layers.Dense(n_hidden, activation='relu'),
    tf.keras.layers.Dense(n_input, activation='sigmoid')
])
# Generate random image using noise
G_output = G(Z)

# Discriminator Network
D = tf.keras.Sequential([
    tf.keras.layers.Dense(n_hidden, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# D(G(Z))
D_gene = D(G_output)

# D(X)
D_real = D(X)

def discriminator_loss(D_real, D_gene):
    return -tf.reduce_mean(tf.math.log(D_real) + tf.math.log(1 - D_gene))

def generator_loss(D_gene):
    return -tf.reduce_mean(tf.math.log(D_gene))

D_var_list = D.trainable_variables
G_var_list = G.trainable_variables

train_D = tf.keras.optimizers.Adam(learning_rate)
train_G = tf.keras.optimizers.Adam(learning_rate)

loss_val_D, loss_val_G = 0, 0

In [4]:
# Training
for epoch in range(total_epoch):
    for step in range(steps):
        batch_xs = train_images_batch[step]
        batch_ys = train_labels_batch[step]
        noise = np.random.normal(size=(batch_size, n_noise))

        with tf.GradientTape() as tape_D, tf.GradientTape() as tape_G:
            D_real = D(batch_xs)
            G_output = G(noise)
            D_gene = D(G_output)

            loss_D = discriminator_loss(D_real, D_gene)
            loss_G = generator_loss(D_gene)

        grads_D = tape_D.gradient(loss_D, D_var_list)
        grads_G = tape_G.gradient(loss_G, G_var_list)

        train_D.apply_gradients(zip(grads_D, D_var_list))
        train_G.apply_gradients(zip(grads_G, G_var_list))

        # loss_val_D += loss_D
        # loss_val_G += loss_G

    print('Epoch', '%04d' % (epoch + 1),
          'D loss: {:.4}'.format(loss_D.numpy() / steps),
          'G loss: {:.4}'.format(loss_G.numpy() / steps))

Epoch 0001 D loss: 0.0001688 G loss: 0.00525
Epoch 0002 D loss: 8.325e-05 G loss: 0.006097
Epoch 0003 D loss: 5.524e-05 G loss: 0.006956
Epoch 0004 D loss: 3.23e-05 G loss: 0.009382
Epoch 0005 D loss: 2.548e-05 G loss: 0.009056
Epoch 0006 D loss: 1.245e-05 G loss: 0.009691
Epoch 0007 D loss: 0.0001629 G loss: 0.005301
Epoch 0008 D loss: 4.076e-05 G loss: 0.008945
Epoch 0009 D loss: 7.193e-05 G loss: 0.006833
Epoch 0010 D loss: 0.0001635 G loss: 0.007275
Epoch 0011 D loss: 0.0002158 G loss: 0.006024
Epoch 0012 D loss: 0.0002224 G loss: 0.00624
Epoch 0013 D loss: 0.0001804 G loss: 0.005582
Epoch 0014 D loss: 0.0002786 G loss: 0.006686
Epoch 0015 D loss: 0.0003972 G loss: 0.006014
Epoch 0016 D loss: 9.94e-05 G loss: 0.007737
Epoch 0017 D loss: 0.0002147 G loss: 0.008151
Epoch 0018 D loss: 0.0003521 G loss: 0.00651
Epoch 0019 D loss: 0.0001736 G loss: 0.008338
Epoch 0020 D loss: 0.0002989 G loss: 0.006294
Epoch 0021 D loss: 0.000338 G loss: 0.005705
Epoch 0022 D loss: 0.0001208 G loss: 0.0

In [5]:
from numpy.random import sample
if epoch == 0 or (epoch + 1) % 10 == 0:
    sample_size = 10
    noise = np.random.normal(size=(sample_size, n_noise))
    samples = G.predict(noise)

    fig, ax = plt.subplots(2, sample_size, figsize=(sample_size, 2))

    for i in range(sample_size):
        ax[0][1].set_axis_off()
        ax[1][1].set_axis_off()

        ax[0][1].imshow(np.reshape(test_images_batch[i], (28, 28)))
        ax[1][1].imshow(np.reshape(samples[i], (28, 28)))

    plt.savefig('samples/{}.png'.format(str(epoch+1).zfill(3)), bbox_inches='tight')
    plt.close(fig)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step




---


# GAN Using Pytorch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.autograd import Variable
from torchvision.utils import save_image

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
bs = 100

In [9]:
# MNIST Dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    # transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))])
    transforms.Normalize(mean=(0.5,), std=(0.5,))])

train_dataset = datasets.MNIST(root='./mnist_data/', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='./mnist_data/', train=False, transform=transform, download=True)

# Data Loader (Input Pipeline)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=bs, shuffle=True)
test_loader = torch.utils.data.DataLoader(dataset=test_dataset,batch_size=bs, shuffle=False)

In [13]:
# Generator
class Generator(nn.Module):
    def __init__(self, g_input_dim, g_output_dim):
        super(Generator, self).__init__()
        self.fc1 = nn.Linear(g_input_dim, 256)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features*2)
        self.fc3 = nn.Linear(self.fc2.out_features, self.fc2.out_features*2)
        self.fc4 = nn.Linear(self.fc3.out_features, g_output_dim)

    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.leaky_relu(self.fc3(x), 0.2)
        return torch.tanh(self.fc4(x))

# Discriminator
class Discriminator(nn.Module):
    def __init__(self, d_input_dim):
        super(Discriminator, self).__init__()
        self.fc1 = nn.Linear(d_input_dim, 1024)
        self.fc2 = nn.Linear(self.fc1.out_features, self.fc1.out_features//2)
        self.fc3 = nn.Linear(self.fc2.out_features, self.fc2.out_features//2)
        self.fc4 = nn.Linear(self.fc3.out_features, 1)

    # forward method
    def forward(self, x):
        x = F.leaky_relu(self.fc1(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc2(x), 0.2)
        x = F.dropout(x, 0.3)
        x = F.leaky_relu(self.fc3(x), 0.2)
        x = F.dropout(x, 0.3)
        return torch.sigmoid(self.fc4(x))

In [14]:
# build network
z_dim = 100
mnist_dim = train_dataset.train_data.size(1) * train_dataset.train_data.size(2)

G = Generator(g_input_dim = z_dim, g_output_dim = mnist_dim).to(device)
D = Discriminator(mnist_dim).to(device)

# loss
criterion = nn.BCELoss()

# optimizer
lr = 0.0002
G_optimizer = optim.Adam(G.parameters(), lr = lr)
D_optimizer = optim.Adam(D.parameters(), lr = lr)


def D_train(x):
    # =================Train the discriminator================= #
    D.zero_grad()

    # train discriminator on real
    x_real, y_real = x.view(-1, mnist_dim), torch.ones(bs, 1)
    x_real, y_real = Variable(x_real.to(device)), Variable(y_real.to(device))

    D_output = D(x_real)
    D_real_loss = criterion(D_output, y_real)
    D_real_score = D_output

    # train discriminator on fake
    z = Variable(torch.randn(bs, z_dim).to(device))
    x_fake, y_fake = G(z), Variable(torch.zeros(bs, 1).to(device))

    D_output = D(x_fake)
    D_fake_loss = criterion(D_output, y_fake)
    D_fake_score = D_output

    # gradient backprop & optimize ONLY D's parameters
    D_loss = D_real_loss + D_fake_loss
    D_loss.backward()
    D_optimizer.step()

    return D_loss.data.item()


def G_train(x):
    # =================Train the generator================= #
    G.zero_grad()

    z = Variable(torch.randn(bs, z_dim).to(device))
    y = Variable(torch.ones(bs, 1).to(device))

    G_output = G(z)
    D_output = D(G_output)
    G_loss = criterion(D_output, y)

    # Gradient backprop & optimize ONLY G's parameters
    G_loss.backward()
    G_optimizer.step()

    return G_loss.data.item()

In [17]:
n_epoch = 200
for epoch in range(1, n_epoch+1):
    D_losses, G_losses = [], []
    for batch_idx, (x, _) in enumerate(train_loader):
        D_losses.append(D_train(x))
        G_losses.append(G_train(x))

    print('[%d/%d]: loss_d: %.3f, loss_g: %.3f' % (
        (epoch), n_epoch, torch.mean(torch.FloatTensor(D_losses)),
        torch.mean(torch.FloatTensor(G_losses))))

    with torch.no_grad():
        test_z = Variable(torch.randn(bs, z_dim).to(device))
        generated = G(test_z)


        save_image(generated.view(generated.size(0), 1, 28, 28), f'./samples/sample_{epoch:03d}.png')

[1/200]: loss_d: 0.943, loss_g: 2.292
[2/200]: loss_d: 0.896, loss_g: 1.987
[3/200]: loss_d: 0.637, loss_g: 2.487
[4/200]: loss_d: 0.445, loss_g: 2.923
[5/200]: loss_d: 0.539, loss_g: 2.620
[6/200]: loss_d: 0.558, loss_g: 2.626
[7/200]: loss_d: 0.568, loss_g: 2.538
[8/200]: loss_d: 0.613, loss_g: 2.435
[9/200]: loss_d: 0.679, loss_g: 2.203
[10/200]: loss_d: 0.668, loss_g: 2.247
[11/200]: loss_d: 0.715, loss_g: 2.145
[12/200]: loss_d: 0.730, loss_g: 2.109
[13/200]: loss_d: 0.764, loss_g: 1.995
[14/200]: loss_d: 0.789, loss_g: 1.924
[15/200]: loss_d: 0.814, loss_g: 1.855
[16/200]: loss_d: 0.793, loss_g: 1.913
[17/200]: loss_d: 0.791, loss_g: 1.865
[18/200]: loss_d: 0.881, loss_g: 1.697
[19/200]: loss_d: 0.878, loss_g: 1.659
[20/200]: loss_d: 0.890, loss_g: 1.634
[21/200]: loss_d: 0.876, loss_g: 1.668
[22/200]: loss_d: 0.942, loss_g: 1.527
[23/200]: loss_d: 0.948, loss_g: 1.514
[24/200]: loss_d: 0.966, loss_g: 1.476
[25/200]: loss_d: 0.970, loss_g: 1.464
[26/200]: loss_d: 1.008, loss_g: 1